# How to use Florence-2

This notebook shows how to use **Florence-2**, the foundational vision model from Microsoft, and is associated with the article **[Florence-2: How it works and how to use it](https://www.assemblyai.com/blog/florence-2-how-it-works-how-to-use)**.

**It is recommended to run this notebook in Colab - if you are coming from the [associated repository](https://github.com/AssemblyAI-Community/florence-2), you can find the link to the colab [here](https://colab.research.google.com/drive/1gD56EBmQ7MZfBkyhGtcrvHtaD2KMSm6W?usp=sharing)**.

If you run this locally you will also need to install the packages in the requirements file.



## Setup

First, we need to install packages beyond those pre-installed in Colab in order to run Florence-2.

In [15]:
!pip install gradio pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 12.3 MB/s eta 0:00:00


In [1]:
%%capture
!pip install timm flash_attn einops;

Next, we clone the associated repository and move the files into the working directory.

In [2]:
%%bash
git clone https://github.com/AssemblyAI-Community/florence-2
mv florence-2/** .
rm -rf ./florence-2/

Cloning into 'florence-2'...


Now we import the packages we'll need, including the `utils.py` module from the repository that we just cloned. This file provides misellaneous functionality to make it easier to work with Florence-2.

In [3]:
import copy

from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image
import requests

import utils

%matplotlib inline

Next we load the Florence-2 model and processor

In [10]:
model_id = 'microsoft/Florence-2-large-ft'
model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True).eval().cuda()
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

config.json:   0%|          | 0.00/2.44k [00:00<?, ?B/s]

configuration_florence2.py:   0%|          | 0.00/15.1k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large-ft:
- configuration_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_florence2.py:   0%|          | 0.00/127k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large-ft:
- modeling_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large-ft:
- processing_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/34.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


And then we set these models as constants for our `utils.py` module so that the functions can utilize them as global constants.

In [11]:
utils.set_model_info(model, processor)

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Character recognition

Character recognition (OCR) tasks detect the text present in images, and optionally return bounding boxes for the identified text.

See also [vocab detection](#scrollTo=ouid-bVL53hd&line=1&uniqifier=1) to actively search for the presence of a specific word or phrase.

In [13]:
path = "/content/drive/MyDrive/PS2-Samples-HackRX5/PS2-Samples-HackRX5/Sample1.png"
image = Image.open(path)
image_rgb = Image.open(path).convert("RGB")

In [14]:
task = utils.TaskType.OCR
results = utils.run_example(task, image_rgb)
print('Detected Text: ', results[task])

Detected Text:  C.Nature of illness / Disease with presenting complaint:CHRONIC ANER DISEASE; PEDE EDEMS, ORANIPSD.Relevant Critical Findings:JANDICE, REDA ENEMAE.Duration of the present alignment:1 year.Daysi.Date of First consultation:81124.(DD/MM/YYYY)ii.Past history of present alignment, if anyDecked as CD & Vect go elsewhere.F.Provisional diagnosis:DECOMPENSAREDCHRONIL NIVER DIRGALE.i.ICD 10 codeG.Proposed line of treatment:i.Medical Managementi.Surgical Managementii.ii.Intensive careiv.Investigationv.Non-allopathic treatment( )H.If investigation and / or Medical Management, provide detailsi.Route of Drug Administration :1.If surgical, name of surgeryLIVERDRANCRANIMATION.I.ICD 10 PCS codeJ.If other treatment, provide detailsK.How did injury occur


In [16]:
import gradio as gr
from PIL import Image
import utils

# OCR processing function
def extract_medical_info(image):
    # Convert image to RGB
    image_rgb = image.convert("RGB")
    # Specify the OCR task type
    task = utils.TaskType.OCR
    # Run the OCR on the image
    results = utils.run_example(task, image_rgb)
    # Extracted text (assuming results[task] contains the detected text)
    detected_text = results.get(task, "No text detected.")
    return detected_text

# Create a Gradio interface
iface = gr.Interface(
    fn=extract_medical_info,  # The function to process the image
    inputs=gr.Image(type="pil"),  # Image input for uploading medical records
    outputs="text",  # Text output to display OCR results
    title="Medical Record OCR Extractor",
    description="Upload an image of a medical record to extract the detailed text information."
)

# Launch the interface
if __name__ == "__main__":
    iface.launch()


Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://02e6eccd5c4d8fc6d8.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
